In [150]:
import numpy as np
import pandas as pd
from itertools import combinations
from scipy.optimize import differential_evolution

In [151]:
theta = {
    "GSL": {"MT": 0.000531, "X": 0.000122, "AUT": 0.000297, "Y": 0.000063},
    "SSL": {"MT": 0.001090, "X": 0.000199, "AUT": 0.000484, "Y": 0.000112}
        }
mu_AUT = 5e-9

# inheritance multipliers for diversity
inheritance = {
        "AUT": 4,
        "X":   3,
        "Y":   2,
        "MT":  2
    }
# mutation rate ratios relative to autosomes
mu_ratio = {
        "AUT": 1.0, #1
        "X":   0.7, #0.7
        "Y":   1.5, #1.5
        "MT":  10.0 #10
    }

PAIR_KEYS = [("MT", "Y"), ("MT", "AUT"), ("MT", "X"), ("AUT", "X")]
COMP_ORDER = ["MT", "Y", "X", "AUT"]


In [152]:
# theoretical piX/piAUT = 0.75
# usually empirical piX/piAUT = 0.5
for sp in theta:
    print(f"{sp}: {theta[sp]['X'] / theta[sp]['AUT']}")

GSL: 0.41077441077441074
SSL: 0.41115702479338845


# functions

In [153]:
def get_Ne_per_component(dNe):
    if all(k in dNe for k in ["MT", "Y"]):
        NeMT = NeF = dNe["MT"]
        NeY = NeM = dNe["Y"]
        NeX = (9 * NeF * NeM) / (2 * NeF + 4 * NeM)
        NeAUT = (4 * NeF * NeM) / (NeF + NeM)
    elif all(k in dNe for k in ["MT", "AUT"]):
        NeMT = NeF = dNe["MT"]
        NeAUT = dNe["AUT"]
        NeY = NeM = (NeMT * NeAUT) / (4 * NeMT - NeAUT)
        NeX = (9 * NeF * NeM) / (2 * NeF + 4 * NeM)
    elif all(k in dNe for k in ["MT", "X"]):
        NeMT = NeF = dNe["MT"]
        NeX = dNe["X"]
        NeY = NeM = (2 * NeMT * NeX) / (9 * NeMT - 4 * NeX)
        NeAUT = (4 * NeF * NeM) / (NeF + NeM)
    elif all(k in dNe for k in ["AUT", "X"]):
        NeAUT = dNe["AUT"]
        NeX = dNe["X"]
        NeY = NeM = (2 * NeAUT * NeX) / (16 * NeX - 9 * NeAUT)
        NeMT = NeF = (2 * NeAUT * NeX) / (9 * NeAUT - 8 * NeX)
    else:
        raise ValueError("Invalid keys in dNe. Expected combinations: ['MT', 'Y'], ['MT', 'AUT'], ['MT', 'X'], ['AUT', 'X']")
    return {"MT": NeMT, "Y": NeY, "X": NeX, "AUT": NeAUT}

In [154]:
def calc_direct_Ne(theta_pop, mu_coef, mu_AUT):
    Ne = {}
    for comp in COMP_ORDER:
        Ne[comp] = theta_pop[comp] / (inheritance[comp] * mu_coef[comp] * mu_AUT)
    return Ne

In [155]:
def build_pairwise_reconstructions(Ne_direct):
    out = {}
    for pair in PAIR_KEYS:
        pair_dict = {k: Ne_direct[k] for k in pair}
        out["_".join(pair)] = get_Ne_per_component(pair_dict)
    return out

In [156]:
def summarize_population(pop, theta_pop, mu_coef, mu_AUT):
    Ne_direct = calc_direct_Ne(theta_pop, mu_coef, mu_AUT)
    recon = build_pairwise_reconstructions(Ne_direct)

    df = pd.DataFrame(index=COMP_ORDER)

    for pair_name, vals in recon.items():
        df[pair_name] = [vals[c] for c in COMP_ORDER]

    df["theta_direct"] = [Ne_direct[c] for c in COMP_ORDER]

    return df

# Point estimates of Ne per component with a fixed mutation rate

In [157]:
for sp in theta.keys():
  print(f"\n=== {sp} ===")
  display(summarize_population(sp, theta[sp], mu_ratio, mu_AUT).round(6))


=== GSL ===


,MT_Y,MT_AUT,MT_X,AUT_X,theta_direct
MT,5310.000000,5310.000000,5310.000000,8479.260516,5310.000000
Y,4200.000000,12340.140845,93920.985864,6603.909418,4200.000000
X,7320.131291,9832.107232,11619.047619,11619.047619,11619.047619
AUT,9380.441640,14850.000000,20103.415505,14850.000000,14850.000000



=== SSL ===


,MT_Y,MT_AUT,MT_X,AUT_X,theta_direct
MT,10900.000000,10900.000000,10900.000000,13860.411570,10900.000000
Y,7466.666667,13596.907216,18535.355693,10736.372757,7466.666667
X,14177.032258,17507.522124,18952.380952,18952.380952,18952.380952
AUT,17724.863884,24200.000000,27454.789969,24200.000000,24200.000000


In [158]:
for sp in theta.keys():
    print(f"\n {sp}")
    print(f"-"*50)
    df = summarize_population(sp, theta[sp], mu_ratio, mu_AUT).round(6)

    for col in df.columns[:-1]:
        nef=df[col]["MT"]
        nem=df[col]["Y"]
        print(f"{col}: \t NeF={nef:.0f}; \t NeM={nem:.0f}; \t r(F/M)={nef/nem:.2f}")


 GSL
--------------------------------------------------
MT_Y: 	 NeF=5310; 	 NeM=4200; 	 r(F/M)=1.26
MT_AUT: 	 NeF=5310; 	 NeM=12340; 	 r(F/M)=0.43
MT_X: 	 NeF=5310; 	 NeM=93921; 	 r(F/M)=0.06
AUT_X: 	 NeF=8479; 	 NeM=6604; 	 r(F/M)=1.28

 SSL
--------------------------------------------------
MT_Y: 	 NeF=10900; 	 NeM=7467; 	 r(F/M)=1.46
MT_AUT: 	 NeF=10900; 	 NeM=13597; 	 r(F/M)=0.80
MT_X: 	 NeF=10900; 	 NeM=18535; 	 r(F/M)=0.59
AUT_X: 	 NeF=13860; 	 NeM=10736; 	 r(F/M)=1.29


# Optimisation of mutation rate ranges

In [159]:
# sp = "SSL"
# theta = {sp: theta[sp]} 
rmu_coef = {
    "MT": [10, 20],
    "Y": [1.5, 2.5],
    "X": [0.7, 0.85],
    "AUT": [1, 1],
}

def objective(params, theta, mu_AUT, use_log_error=False, verbose=False):
    mu_coef = {
        "MT": params[0],
        "Y": params[1],
        "X": params[2],
        "AUT": 1.0,
    }

    total_error = 0.0
    penalty = 1e12

    for pop, theta_pop in theta.items():
        try:
            Ne_direct = calc_direct_Ne(theta_pop, mu_coef, mu_AUT)
            recon = build_pairwise_reconstructions(Ne_direct)

            for pair_name, rec_vals in recon.items():
                for comp in COMP_ORDER:
                    target = Ne_direct[comp]
                    pred = rec_vals[comp]

                    if target <= 0 or pred <= 0 or not np.isfinite(target) or not np.isfinite(pred):
                        return penalty

                    if use_log_error:
                        err = (np.log(pred) - np.log(target)) ** 2
                    else:
                        err = ((pred - target) / target) ** 2

                    total_error += err

        except Exception:
            return penalty

    return total_error

bounds = [
    tuple(rmu_coef["MT"]),
    tuple(rmu_coef["Y"]),
    tuple(rmu_coef["X"]),
]

result = differential_evolution(
    objective,
    bounds=bounds,
    args=(theta, mu_AUT, True),
    seed=42,
    polish=True,
    tol=1e-8,
)

best_mu_coef = {
    "MT": result.x[0],
    "Y": result.x[1],
    "X": result.x[2],
    "AUT": 1.0,
}

print("Optimization success:", result.success)
print("Best mu_coef:")
for k, v in best_mu_coef.items():
    print(f"  {k}: {v:.6f}")

# for pop, theta_pop in theta.items():
#     print(f"\n=== {pop} ===")
#     display(summarize_population(pop, theta_pop, best_mu_coef, mu_AUT).round(6))

Optimization success: True
Best mu_coef:
  MT: 10.000000
  Y: 1.500000
  X: 0.808545
  AUT: 1.000000


# NeF and NeM optimiser based on nucleotide diversity

In [160]:
import numpy as np
from scipy.optimize import minimize


def objective(pi_obs, mu_ratio, mu_AUT, inheritance):

    def Ne_values(Nm, Nf):
        Ne_AUT = 4*Nm*Nf/(Nm + Nf)
        Ne_X   = 9*Nm*Nf/(4*Nm + 2*Nf)
        Ne_Y   = Nm
        Ne_MT  = Nf

        return {
            "AUT": Ne_AUT,
            "X":   Ne_X,
            "Y":   Ne_Y,
            "MT":  Ne_MT
        }

    def predicted_pi(Nm, Nf):
        Ne = Ne_values(Nm, Nf)
        pi_pred = {}

        for comp in ["AUT","X","Y","MT"]:
            pi_pred[comp] = inheritance[comp] * Ne[comp] * mu_ratio[comp] * mu_AUT

        return pi_pred

    def loss(params):
        Nm, Nf = params
        if Nm <= 0 or Nf <= 0:
            return 1e12

        pi_pred = predicted_pi(Nm, Nf)

        err = 0
        for comp in pi_obs:
            err += (np.log(pi_obs[comp]) - np.log(pi_pred[comp]))**2

        return err

    # initial guess
    x0 = [100000,100000]

    res = minimize(loss, x0, method="Nelder-Mead")

    Nm_est, Nf_est = res.x

    print("Estimated Nm:", Nm_est)
    print("Estimated Nf:", Nf_est)
    print("Sex ratio Nf/Nm:", Nf_est/Nm_est)

# run the optimisation for each species
for sp in theta.keys():
    print(f"\n=== {sp} ===")

    # observed nucleotide diversity values
    pi_obs = theta[sp]

    objective(pi_obs, mu_ratio, mu_AUT, inheritance)


=== GSL ===
Estimated Nm: 5233.162047338714
Estimated Nf: 6752.100624920439
Sex ratio Nf/Nm: 1.290252540212121

=== SSL ===
Estimated Nm: 8708.993803387795
Estimated Nf: 12624.66483426953
Sex ratio Nf/Nm: 1.44961233401711
